# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 9: Bezugsdauer

Letztes (nachrangiges) Themenfeld aus [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md): wie lange und in welcher Haushaltskonstellation Menschen im SGB-II-Bezug stehen — kombiniert **drei** Tabellenblätter, direkt aus den Excel-Rohdaten gelesen (wie schon die Kapitel 6–8):
- **„3.1 Bedarfsgemeinschaften“**: Struktur nach Haushaltsgröße und BG-Typ.
- **„3.4 Langzeitleistungsbezug“**: Langzeitleistungsbeziehende (LZB) — Personen, die in den vergangenen 24 Monaten mindestens 21 Monate leistungsberechtigt waren — im Vergleich zu Langzeitarbeitslosen (LZA).
- **„3.5 Bewegungen Personen“**: Zu- und Abgänge im Regelleistungsbezug, inkl. „Drehtür-Effekt“ (erneuter Bezug kurz nach Abgang).

⚠️ **Wartezeit:** wie bei den Kapiteln 6–8 nur ein Datenblock mit **3 Monaten Wartezeit** → Zeitreihe für **Oktober 2024 – Juni 2026**.

In [1]:
from pathlib import Path

import openpyxl
import pandas as pd
import plotly.graph_objects as go

RAW_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"


def parse_bezugsdauer(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws1 = wb["3.1 Bedarfsgemeinschaften"]
    ws4 = wb["3.4 Langzeitleistungsbezug"]
    ws5 = wb["3.5 Bewegungen Personen"]
    monat = pd.Period(ws1["B32"].value, freq="M")
    zeile = {
        "period": monat,
        "BG Insgesamt": ws1["B36"].value,
        "BG 1 Person": ws1["B37"].value,
        "BG 2 Personen": ws1["B38"].value,
        "BG 3 Personen": ws1["B39"].value,
        "BG 4 Personen": ws1["B40"].value,
        "BG 5+ Personen": ws1["B41"].value,
        "Single-BG": ws1["B44"].value,
        "Alleinerziehende-BG": ws1["B45"].value,
        "Partner-BG ohne Kinder": ws1["B49"].value,
        "Partner-BG mit Kindern": ws1["B50"].value,
        "ELB Insgesamt": ws4["B36"].value,
        "LZB": ws4["B37"].value,
        "LZA (3.4)": ws4["B48"].value,
        "Zugang ELB": ws5["B37"].value,
        "Anteil Zugang mit Vorbezug (%)": ws5["E38"].value,
        "Abgang ELB": ws5["B48"].value,
        "Drehtür-Quote (%)": ws5["E49"].value,
    }
    wb.close()
    return zeile


bezugsdauer = pd.DataFrame(
    [parse_bezugsdauer(p) for p in sorted(RAW_DIR.glob("*.xlsx"))]
).set_index("period").sort_index()
bezugsdauer.tail()

ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"


def export_abbildung(fig, dateiname, width, height, **layout):
    """Speichert eine Kopie der Grafik im weissen Layout der Projektdokumentation
    (Docs/Abbildungen, doppelte Aufloesung); die Anzeige im Notebook bleibt unveraendert.
    Optionale `layout`-Angaben (z. B. kuerzerer Titel) gelten nur fuer die exportierte Datei."""
    go.Figure(fig).update_layout(template="plotly_white", **layout).write_image(
        ABBILDUNGEN_DIR / dateiname, scale=2, width=width, height=height
    )

C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


## 1. Datenüberblick und Plausibilitätsprüfung

In [2]:
print(f"Anzahl Monate: {len(bezugsdauer)}")
print(f"Zeitraum: {bezugsdauer.index.min()} bis {bezugsdauer.index.max()}")
print(f"Duplikate: {bezugsdauer.index.duplicated().sum()}")

bg_groessen = ["BG 1 Person", "BG 2 Personen", "BG 3 Personen", "BG 4 Personen", "BG 5+ Personen"]
groessen_check = bezugsdauer[bg_groessen].sum(axis=1) - bezugsdauer["BG Insgesamt"]
print(f"BG-Größen-Summen-Check: min={groessen_check.min()}, max={groessen_check.max()}")

Anzahl Monate: 21
Zeitraum: 2024-10 bis 2026-06
Duplikate: 0
BG-Größen-Summen-Check: min=0, max=0


Alle 21 Dateien liefern 21 lückenlose, eindeutige Monate (Oktober 2024 – Juni 2026, keine Duplikate). Die fünf Haushaltsgrößen-Klassen summieren sich in jedem Monat exakt zur Gesamtzahl der Bedarfsgemeinschaften (Differenz = 0) — die Daten sind konsistent.

## 2. Struktur der Bedarfsgemeinschaften: Größe und Typ

In [3]:
fig = go.Figure()
for spalte in bg_groessen:
    fig.add_trace(go.Scatter(
        x=bezugsdauer.index.to_timestamp(), y=bezugsdauer[spalte], name=spalte,
        mode="lines", stackgroup="groesse", line=dict(width=0.5),
    ))
fig.update_layout(
    title="Bedarfsgemeinschaften nach Haushaltsgröße",
    yaxis_title="Anzahl Bedarfsgemeinschaften", xaxis_title="Monat",
    hovermode="x unified", legend=dict(orientation="h", y=1.15), height=440,
)
export_abbildung(fig, "abb25_bg_groesse.png", 1000, 440)
fig

In [4]:
bg_typen = ["Single-BG", "Alleinerziehende-BG", "Partner-BG ohne Kinder", "Partner-BG mit Kindern"]
fig = go.Figure()
for spalte in bg_typen:
    fig.add_trace(go.Scatter(
        x=bezugsdauer.index.to_timestamp(), y=bezugsdauer[spalte], name=spalte,
        mode="lines+markers", marker=dict(size=5),
    ))
fig.update_layout(
    title="Bedarfsgemeinschaften nach BG-Typ",
    yaxis_title="Anzahl Bedarfsgemeinschaften", xaxis_title="Monat",
    hovermode="x unified", legend=dict(orientation="h", y=1.15), height=440,
)
export_abbildung(fig, "abb26_bg_typ.png", 1000, 440)
fig

Die Zahl der Bedarfsgemeinschaften insgesamt sinkt leicht (2.758 → 2.651), getragen vor allem von einem klaren Strukturtrend: **Single-Bedarfsgemeinschaften** gewinnen relativ an Gewicht (Anteil 57,0 % → 59,4 % — bereits fast 6 von 10 Bedarfsgemeinschaften bestehen nur aus einer Person), während **Alleinerziehende-BG** (544 → 486) und **Partner-BG** (mit wie ohne Kinder: 604 → 529 zusammen) spürbar zurückgehen. Der SGB-II-Bezug in Regensburg wird damit zunehmend von Einpersonenhaushalten geprägt.

## 3. Langzeitleistungsbezug (LZB) im Vergleich zur Langzeitarbeitslosigkeit (LZA)

**Langzeitleistungsbeziehende (LZB)** sind erwerbsfähige Leistungsberechtigte, die in den vergangenen 24 Monaten mindestens 21 Monate leistungsberechtigt waren — ein anderes Konzept als **Langzeitarbeitslose (LZA)**: Bürgergeld kann auch ohne Arbeitslosigkeit bezogen werden (z. B. bei Erwerbstätigkeit mit zu geringem Einkommen, siehe [`06_Erwerbstaetigkeit.ipynb`](06_Erwerbstaetigkeit.ipynb)).

In [5]:
anteil_lzb = (bezugsdauer["LZB"] / bezugsdauer["ELB Insgesamt"] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=bezugsdauer.index.to_timestamp(), y=bezugsdauer["LZB"], name="LZB (absolut)", mode="lines+markers", line=dict(color="#2563eb"), marker=dict(size=5), yaxis="y1"))
fig.add_trace(go.Scatter(x=bezugsdauer.index.to_timestamp(), y=anteil_lzb, name="Anteil LZB an allen ELB (%)", mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=5), yaxis="y2"))
fig.add_trace(go.Scatter(x=bezugsdauer.index.to_timestamp(), y=bezugsdauer["LZA (3.4)"], name="LZA (zum Vergleich)", mode="lines+markers", line=dict(color="#dc2626"), marker=dict(size=5), yaxis="y1"))
fig.update_layout(
    title="Langzeitleistungsbezug (LZB) vs. Langzeitarbeitslosigkeit (LZA)",
    yaxis=dict(title="Personen"),
    yaxis2=dict(title="Anteil LZB an ELB (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis_title="Monat", hovermode="x unified", legend=dict(orientation="h", y=1.15), height=460,
)
export_abbildung(fig, "abb27_lzb_lza.png", 1000, 460)
fig

Der Anteil der Langzeitleistungsbeziehenden (LZB) an allen erwerbsfähigen Leistungsberechtigten steigt deutlich von 48,9 % auf 56,3 % (Spanne 48,9–56,6 %) — **über die Hälfte** aller ELB bezieht mittlerweile seit mindestens 21 der letzten 24 Monate Leistungen. Die absolute LZB-Zahl steigt dabei sogar (1.785 → 1.951, Höchststand 1.973 im Mai 2026), obwohl die Gesamtzahl der ELB sinkt (3.648 → 3.465) — der Bezug verfestigt sich also für einen wachsenden Teil der verbleibenden Leistungsberechtigten. Zum Vergleich: die (separat erfasste) Langzeitarbeitslosigkeit steigt im selben Zeitraum noch deutlicher (461 → 709 Personen in diesem Blatt) — beide Kennzahlen zeigen unabhängig voneinander dieselbe Verfestigungstendenz, messen aber unterschiedliche Dinge (Leistungsbezug vs. Arbeitslosigkeit).

## 4. Bewegungsdynamik: der „Drehtür-Effekt“

Zwei Kennzahlen aus Blatt „3.5“: der Anteil der **Zugänge**, die bereits zuvor im Leistungsbezug waren (statt zum ersten Mal), und die **Drehtür-Quote** — der Anteil der **Abgänge**, die innerhalb von 3 Monaten erneut in den Leistungsbezug zurückkehren.

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=bezugsdauer.index.to_timestamp(), y=bezugsdauer["Anteil Zugang mit Vorbezug (%)"], name="Zugang mit Vorbezug (%)", mode="lines+markers", line=dict(color="#7c3aed"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=bezugsdauer.index.to_timestamp(), y=bezugsdauer["Drehtür-Quote (%)"], name="Drehtür-Quote: erneuter Bezug binnen 3 Monaten (%)", mode="lines+markers", line=dict(color="#dc2626"), marker=dict(size=5)))
fig.update_layout(
    title="Bewegungsdynamik im Regelleistungsbezug (ELB)",
    yaxis_title="Prozent", yaxis_ticksuffix="%",
    xaxis_title="Monat", hovermode="x unified", legend=dict(orientation="h", y=1.15), height=440,
)
export_abbildung(fig, "abb28_drehtuer.png", 1000, 440)
fig

Beide Kennzahlen bestätigen den Befund einer sich verfestigenden Bedarfslage: Der **Anteil der Zugänge mit Vorbezug** steigt deutlich von 56,1 % auf 70,9 % (Ø 70,2 %, Spanne 56,1–79,3 % mit Höchstwert im Dezember 2025) — mittlerweile haben gut zwei von drei „Neuzugängen“ bereits früher einmal Leistungen bezogen, echte Erstzugänge werden seltener. Die **Drehtür-Quote** (Anteil der Abgänge, die binnen 3 Monaten zurückkehren) schwankt stark zwischen 17,8 % und 33,5 % (Ø 25,4 %) ohne klaren Trend — im Schnitt kehrt aber etwa jede vierte Person, die den Leistungsbezug verlässt, innerhalb eines Vierteljahres zurück.

## 5. Export der aufbereiteten Daten

Grundlage für den Dashboard-Tab „⏳ Bezugsdauer“.

In [7]:
bezugsdauer.to_csv(PROCESSED_DIR / "bezugsdauer_sgb2.csv")
bezugsdauer.tail()

,BG Insgesamt,BG 1 Person,BG 2 Personen,BG 3 Personen,BG 4 Personen,BG 5+ Personen,Single-BG,Alleinerziehende-BG,Partner-BG ohne Kinder,Partner-BG mit Kindern,ELB Insgesamt,LZB,LZA (3.4),Zugang ELB,Anteil Zugang mit Vorbezug (%),Abgang ELB,Drehtür-Quote (%)
period,,,,,,,,,,,,,,,,,
2026-02,2655,1557,459,262,170,207,1554,490,187,369,3537,1955,662,187,70.053476,124,25.806452
2026-03,2696,1599,461,256,176,204,1598,480,188,375,3569,1966,677,188,73.404255,152,17.763158
2026-04,2681,1589,467,251,175,199,1589,482,187,370,3532,1962,694,167,75.449102,200,26.500000
2026-05,2687,1601,467,244,177,198,1598,484,183,365,3519,1973,714,150,72.000000,160,21.875000
2026-06,2651,1579,473,237,175,187,1576,486,180,349,3465,1951,709,141,70.921986,193,23.316062


## 6. Zusammenfassung

- **Haushaltsstruktur:** Single-Bedarfsgemeinschaften nehmen relativ zu (57,0 % → 59,4 %), Alleinerziehende- und Partner-BG gehen zurück — der SGB-II-Bezug wird zunehmend von Einpersonenhaushalten geprägt.
- **Langzeitleistungsbezug:** über die Hälfte aller ELB (48,9 % → 56,3 %) bezieht mittlerweile seit mindestens 21 der letzten 24 Monate Leistungen — trotz sinkender ELB-Gesamtzahl steigt die absolute LZB-Zahl.
- **Bewegungsdynamik:** der Anteil der Zugänge mit Vorbezug steigt deutlich (56,1 % → 70,9 %) — echte Erstzugänge werden seltener. Die Drehtür-Quote schwankt stark (Ø 25,4 %) ohne klaren Trend.
- **Gesamtbild:** Alle drei Blätter zeigen unabhängig voneinander dieselbe Tendenz wie die bereits in [`03_Statistik.ipynb`](03_Statistik.ipynb) beschriebene Verschärfung der Langzeitarbeitslosigkeit — der SGB-II-Bezug in Regensburg verfestigt sich für einen wachsenden Teil der Betroffenen, auch wenn die absoluten Fallzahlen insgesamt leicht rückläufig sind.
- **Datenbesonderheit:** Zeitreihe Oktober 2024 – Juni 2026 (3 Monate Wartezeit), analog zu Kapitel 6–8.

Damit sind alle in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md) priorisierten und nachrangigen Themen abgearbeitet. Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py), Tab „⏳ Bezugsdauer“.